# Fixed nested ToolRet candidate pools

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import sys
from pathlib import Path

# 与 notebook00 使用同一个项目目录
ROOT = Path("/content/drive/MyDrive/DiffJev")
assert (ROOT / "AGENTS.md").is_file(), f"项目目录不存在：{ROOT}"

os.chdir(ROOT)
os.environ["DIFFJEV_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from diffjev.notebook_runtime import NotebookRuntime, install_environment

ENV_DIR = Path("/content/diffjev-env")
WORKER = ENV_DIR / "bin/python"

if not WORKER.is_file():
    # 新运行时：只重建依赖环境，不下载模型、不重新准备数据
    runtime = install_environment(ROOT, ENV_DIR)
else:
    runtime = NotebookRuntime(ROOT, WORKER)
    runtime.save_settings()

print("项目目录：", ROOT)
print("实验 Python：", runtime.python)

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists() and Path('/content/drive/MyDrive/DiffJev/AGENTS.md').exists():
    ROOT = Path('/content/drive/MyDrive/DiffJev')
assert (ROOT / 'AGENTS.md').exists(), 'Mount Drive and set DIFFJEV_ROOT to the checkout used by notebook 00'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev.notebook_runtime import NotebookRuntime
runtime = NotebookRuntime(ROOT)
runtime.python_code("from diffjev.preparation import check_environment; from pathlib import Path; check_environment(Path.cwd())", label='05_exp3_scaling_environment')
def run_python(source):
    prefix = '''import gc
from pathlib import Path
from diffjev import workflows as w
from diffjev.config import load_yaml_config
ROOT = w.ROOT
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']
'''
    runtime.python_code(prefix + source, label='05_exp3_scaling')
print('Experiment worker:', runtime.python)


In [ ]:
run_python(r"""
# Reuse the frozen manifests from notebook 01; no negative mining on test outcomes.
for backbone in cfg['backbones']:
    methods = ['diffjev','serialized_dlm'] + (['ar'] if backbone.startswith('nemotron') else [])
    for method in methods:
        runner = w.load_runner(backbone,method=method,seed=SEED,device=DEVICE)
        w.evaluate_scaling(runner,w.run_directory(backbone,method,seed=SEED)/'evaluation')
        del runner
        gc.collect()
""")

In [ ]:
run_python(r"""
for name in cfg['references'] + ['bge']:
    runner=w.baseline_runner(name,device=DEVICE)
    w.evaluate_scaling(runner,ROOT/'outputs/formal/references'/name)
    del runner
    gc.collect()
""")

K > 32 is cardinality extrapolation. Reports include actual K, Hit@1, coverage, raw query predictions, and stagewise gold survival. These are supplied-candidate routing results, not full-corpus retrieval or tool execution.